# Add schema and SQL dialect context

A model cannot inspect our local database. In this notebook, we obtain the `air_traffic` schema from DuckDB, state the database dialect explicitly, and use both details to make the prompt context reusable.

The notebook needs the persistent DuckDB database created in `03a_duckdb_settings.ipynb`. Set `DUCKDB_DATABASES_PATH` before starting Jupyter. The local checkpoint does not make a model request.

In [1]:
import os
import re
from pathlib import Path

import ibis
from openai import OpenAI

## Connect to the prepared database

The previous data-preparation notebook creates `sql-ai-agent.duckdb` and its `air_traffic` table. We reconnect to that same file instead of loading the CSV again.

In [2]:
def connect_to_prepared_database():
    database_directory = os.getenv("DUCKDB_DATABASES_PATH")
    if not database_directory:
        raise EnvironmentError(
            "Set DUCKDB_DATABASES_PATH to the directory used in the data tutorial."
        )

    database_path = Path(database_directory).expanduser() / "sql-ai-agent.duckdb"
    if not database_path.is_file():
        raise FileNotFoundError(
            f"Could not find {database_path}. Run 03a_duckdb_settings.ipynb first."
        )

    try:
        connection = ibis.duckdb.connect(database_path, read_only=True)
    except Exception as error:
        if "Could not set lock" in str(error):
            raise RuntimeError(
                "The DuckDB file is already open for writing. Shut down the kernel "
                "that opened it, then restart this notebook kernel and try again."
            ) from error
        raise
    table_name = "air_traffic"
    if table_name not in connection.list_tables():
        raise RuntimeError(f"Could not find the {table_name!r} table.")

    return connection, table_name, database_path


connection, table_name, database_path = connect_to_prepared_database()
print(f"Connected to {database_path.name}; available table: {table_name}")

Connected to sql-ai-agent.duckdb; available table: air_traffic


## Build schema context from DuckDB metadata

`information_schema.columns` is metadata maintained by the database. Querying it keeps the prompt description aligned with the table, rather than asking us to hand-copy column names and types.

In [3]:
def quote_identifier(identifier):
    return '"' + identifier.replace('"', '""') + '"'


def build_schema_context(connection, table_name):
    if not table_name.isidentifier():
        raise ValueError("This tutorial accepts a simple table name only.")

    schema_sql = f"""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = '{table_name}'
    ORDER BY ordinal_position
    """
    schema_details = connection.sql(schema_sql).execute()
    if schema_details.empty:
        raise RuntimeError(f"No schema metadata found for {table_name!r}.")

    columns = "\n".join(
        f"    {quote_identifier(row.column_name)} {row.data_type}"
        for row in schema_details.itertuples(index=False)
    )
    return f"CREATE TABLE {quote_identifier(table_name)} (\n{columns}\n);"


schema_context = build_schema_context(connection, table_name)
print(schema_context)
assert schema_context.count("\n    ") == 13

CREATE TABLE "air_traffic" (
    "Year" BIGINT
    "Date" TIMESTAMP
    "Operating Airline" VARCHAR
    "Operating Airline IATA Code" VARCHAR
    "Published Airline" VARCHAR
    "Published Airline IATA Code" VARCHAR
    "GEO Summary" VARCHAR
    "GEO Region" VARCHAR
    "Activity Type Code" VARCHAR
    "Price Category Code" VARCHAR
    "Terminal" VARCHAR
    "Boarding Area" VARCHAR
    "Passenger Count" BIGINT
);


## Make the database dialect explicit

Table and column names alone do not tell a model which SQL syntax the database accepts. The small mapping below keeps the database choice configurable and makes DuckDB-specific requirements visible in the resulting prompt.

In [4]:
DIALECT_GUIDANCE = {
    "DuckDB": (
        "Use DuckDB SQL. Double-quote table and column identifiers, "
        "use LIMIT to restrict rows, and do not use SQL Server TOP syntax."
    )
}


def build_dialect_context(dialect):
    try:
        guidance = DIALECT_GUIDANCE[dialect]
    except KeyError as error:
        supported = ", ".join(DIALECT_GUIDANCE)
        raise ValueError(f"Unsupported dialect {dialect!r}. Choose one of: {supported}.") from error

    return f"Database dialect: {dialect}. {guidance}"


database_dialect = "DuckDB"
dialect_context = build_dialect_context(database_dialect)
print(dialect_context)

Database dialect: DuckDB. Use DuckDB SQL. Double-quote table and column identifiers, use LIMIT to restrict rows, and do not use SQL Server TOP syntax.


## Assemble an extensible context-aware prompt

The system prompt receives stable task instructions plus the context derived above. `context_blocks` is empty in this tutorial, but it gives later tutorials one explicit place to add bounded values, skills, or selected conversation context without hiding those additions in the agent function. The user prompt can remain focused on the question.

In [5]:
def build_system_prompt(schema_context, dialect_context, context_blocks=()):
    if any(not isinstance(block, str) or not block.strip() for block in context_blocks):
        raise ValueError("Each context block must be a non-empty string.")

    prompt_sections = [
        "\n".join(
            [
                "Translate the user's question into one read-only SQL query.",
                "Return only SQL as plain text, without Markdown or an explanation.",
                "Use only the table and columns supplied below.",
            ]
        ),
        dialect_context,
        f"Available schema:\n{schema_context}",
    ]
    prompt_sections.extend(context_blocks)
    return "\n\n".join(prompt_sections)


system_prompt = build_system_prompt(schema_context, dialect_context)
print(system_prompt)
assert "DuckDB" in system_prompt
assert '"Passenger Count" BIGINT' in system_prompt

Translate the user's question into one read-only SQL query.
Return only SQL as plain text, without Markdown or an explanation.
Use only the table and columns supplied below.

Database dialect: DuckDB. Use DuckDB SQL. Double-quote table and column identifiers, use LIMIT to restrict rows, and do not use SQL Server TOP syntax.

Available schema:
CREATE TABLE "air_traffic" (
    "Year" BIGINT
    "Date" TIMESTAMP
    "Operating Airline" VARCHAR
    "Operating Airline IATA Code" VARCHAR
    "Published Airline" VARCHAR
    "Published Airline IATA Code" VARCHAR
    "GEO Summary" VARCHAR
    "GEO Region" VARCHAR
    "Activity Type Code" VARCHAR
    "Price Category Code" VARCHAR
    "Terminal" VARCHAR
    "Boarding Area" VARCHAR
    "Passenger Count" BIGINT
);


## Compare missing context with the corrected query

These two deliberate mistakes are ordinary `SELECT` statements, so they are safe to run as demonstrations. DuckDB rejects the nonexistent `Passengers` column and SQL Server's `TOP 1` syntax. The final query uses the inspected column name and DuckDB-compatible syntax.

In [6]:
def show_expected_error(label, sql):
    try:
        connection.sql(sql).execute()
    except Exception as error:
        print(f"{label}: {type(error).__name__}")
    else:
        raise AssertionError(f"{label} unexpectedly ran successfully.")


show_expected_error(
    "Schema mismatch",
    'SELECT SUM("Passengers") FROM "air_traffic" WHERE "Year" = 2024',
)
show_expected_error(
    "Dialect mismatch",
    'SELECT TOP 1 "Passenger Count" FROM "air_traffic"',
)

verified_sql = """
SELECT SUM("Passenger Count") AS "total_passengers"
FROM "air_traffic"
WHERE "Year" = 2024
"""
verified_result = connection.sql(verified_sql).execute()
print(verified_result.to_string(index=False))
assert int(verified_result.iloc[0, 0]) == 52_210_939

Schema mismatch: BinderException
Dialect mismatch: ParserException
total_passengers
        52210939


## Run the schema-aware agent function

`sql_ai_agent()` now owns the complete T04 flow and constructs its schema-and-dialect prompt for each question. Calling it with `context_blocks=()` preserves the T05 checkpoint; later tutorials can pass a bounded data-value block, a skill, or selected conversation context in a visible order. The test client below exercises the full function without an API request.

In [7]:
def extract_sql(text):
    fenced_sql = re.search(
        r"```(?:sql)?\s*(.*?)```",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )
    sql = fenced_sql.group(1) if fenced_sql else text
    sql = sql.strip()
    if not sql:
        raise ValueError("The model did not return a SQL query.")
    return sql


def sql_ai_agent(
    question,
    client,
    connection,
    table_name,
    model="gpt-5.4-mini",
    dialect="DuckDB",
    context_blocks=(),
):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("question must be a non-empty string.")

    schema_context = build_schema_context(connection, table_name)
    dialect_context = build_dialect_context(dialect)
    system_prompt = build_system_prompt(
        schema_context,
        dialect_context,
        context_blocks=context_blocks,
    )
    response = client.responses.create(
        model=model,
        instructions=system_prompt,
        input=f"Write a SQL query that answers: {question}",
    )
    sql = extract_sql(response.output_text)
    result = connection.sql(sql).execute()

    separator = "-" * 72
    print(f"Question:\n{question}\n")
    print(f"Generated SQL:\n{separator}\n{sql}\n{separator}")
    print("\nResult:")
    print(result.to_string(index=False))

    return sql, result


class FakeResponse:
    def __init__(self, output_text):
        self.output_text = output_text


class FakeResponses:
    def __init__(self, output_text):
        self.output_text = output_text
        self.request = None

    def create(self, **kwargs):
        self.request = kwargs
        return FakeResponse(self.output_text)


class FakeClient:
    def __init__(self, output_text):
        self.responses = FakeResponses(output_text)


fake_client = FakeClient(f"```sql\n{verified_sql.strip()}\n```")
generated_sql, agent_result = sql_ai_agent(
    "What was the total passenger count in 2024?",
    client=fake_client,
    connection=connection,
    table_name=table_name,
    model="test-model",
)
assert generated_sql == verified_sql.strip()
assert int(agent_result.iloc[0, 0]) == 52_210_939
assert "DuckDB" in fake_client.responses.request["instructions"]
assert '"Passenger Count" BIGINT' in fake_client.responses.request["instructions"]

if os.getenv("OPENAI_API_KEY"):
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    print("OpenAI client is ready. Call sql_ai_agent() with the live client to make a request.")
else:
    print("Set OPENAI_API_KEY to enable a live model request.")

OpenAI client is ready. Call sql_ai_agent() to make a request.


## Checkpoint

We now have a reusable schema-and-dialect context layer for the DuckDB table. Next, we can supply bounded stored values when column names and types still leave a label ambiguous.